# Rainfall Analysis and Planting Advisory — Five Kenyan Counties

**Goal:** clean hand-typed rainfall data for Machakos, analyse monthly rainfall for five counties with NumPy, and produce a planting advisory for a farmer-support NGO.

**Workflow:** clean data → combine into a 5 × 12 array → totals and averages → wettest month → dry months → planting window → write CSV → advisory.

## 1. Data and imports

In [1]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = np.array(["January", "February", "March", "April", "May", "June",
                   "July", "August", "September", "October", "November", "December"])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## 2. Clean the Machakos row

`clean_rainfall()` tries to convert each entry with `int()`. A bad entry such as `"N/A"` or an empty string raises a `ValueError`, which is caught specifically (no bare `except`). Invalid months are filled with the rounded mean of the valid Machakos values, and the imputed months are reported.

`int(" 20 ")` succeeds because `int()` ignores surrounding whitespace, so that value is kept as a real reading (20), not imputed.

In [2]:
def clean_rainfall(raw_values):
    """Convert raw text readings to integers.

    Returns (cleaned_values, imputed_indices). Invalid entries are replaced
    with the rounded mean of the valid entries.
    """
    parsed = []
    invalid_idx = []
    for i, value in enumerate(raw_values):
        try:
            parsed.append(int(value))
        except ValueError:
            parsed.append(None)
            invalid_idx.append(i)

    valid = [v for v in parsed if v is not None]
    fill_value = round(sum(valid) / len(valid))
    cleaned = [fill_value if v is None else v for v in parsed]
    return cleaned, invalid_idx, fill_value


machakos_clean, imputed_idx, fill_value = clean_rainfall(machakos_raw)

print("Cleaned Machakos row:", machakos_clean)
print(f"Fill value (rounded mean of valid readings): {fill_value} mm")
print("Imputed months:", [str(months[i]) for i in imputed_idx])
for i in imputed_idx:
    print(f"  {months[i]}: raw value {machakos_raw[i]!r} -> {machakos_clean[i]} mm")

Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Fill value (rounded mean of valid readings): 68 mm
Imputed months: ['April', 'September']
  April: raw value 'N/A' -> 68 mm
  September: raw value '' -> 68 mm


## 3. Combine into a 5 × 12 NumPy array

In [3]:
R = np.array(clean_rows + [machakos_clean])

print("Shape:", R.shape)
print("dtype:", R.dtype)
assert R.shape == (5, 12)
R

Shape: (5, 12)
dtype: int64


array([[ 64,  56,  92, 219, 176,  35,  18,  24,  31,  61, 165, 104],
       [ 35,  42,  70, 145, 120,  78,  96, 105,  72,  55,  60,  38],
       [ 58,  90, 150, 240, 180,  95,  70,  85,  90,  88, 130,  95],
       [ 30,  15,  55, 150, 240, 110,  90,  65,  60,  90, 100,  70],
       [ 48,  35,  88,  68, 140,  20,   9,  12,  68,  45, 182,  96]])

## 4. Annual totals and monthly averages

`axis=1` collapses the months (one total per county); `axis=0` collapses the counties (one average per month).

In [4]:
annual_totals = R.sum(axis=1)
monthly_avg = R.mean(axis=0)

print("Annual total rainfall (mm):")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<9} {total}")

print("\nAverage rainfall per month across all five counties (mm):")
for month, avg in zip(months, monthly_avg):
    print(f"  {month:<9} {avg:.1f}")

Annual total rainfall (mm):
  Nairobi   1045
  Nakuru    916
  Kisumu    1371
  Mombasa   1075
  Machakos  811

Average rainfall per month across all five counties (mm):
  January   47.0
  February  47.6
  March     91.0
  April     164.4
  May       171.2
  June      67.6
  July      56.6
  August    58.2
  September 64.2
  October   67.8
  November  127.4
  December  80.6


## 5. Wettest month per county (`np.argmax`, `axis=1`)

In [5]:
wettest_idx = np.argmax(R, axis=1)
wettest_months = months[wettest_idx]

for county, month, mm in zip(counties, wettest_months, R.max(axis=1)):
    print(f"  {county:<9} {month:<9} ({mm} mm)")

  Nairobi   April     (219 mm)
  Nakuru    April     (145 mm)
  Kisumu    April     (240 mm)
  Mombasa   May       (240 mm)
  Machakos  November  (182 mm)


## 6. Dry months (below 50 mm) using a boolean mask

In [6]:
dry_mask = R < 50                 # True where the month is dry
dry_counts = dry_mask.sum(axis=1) # True counts as 1

for county, n in zip(counties, dry_counts):
    print(f"  {county:<9} {n} dry month(s)")

  Nairobi   4 dry month(s)
  Nakuru    3 dry month(s)
  Kisumu    0 dry month(s)
  Mombasa   2 dry month(s)
  Machakos  6 dry month(s)


## 7. Planting window: best two consecutive months

`R[:, :-1] + R[:, 1:]` adds each month to the next one, giving 11 two-month totals per county (Jan+Feb, Feb+Mar, ... Nov+Dec). `argmax` on each row gives the start month of the wettest pair.

In [7]:
pair_totals = R[:, :-1] + R[:, 1:]          # shape (5, 11)
start_idx = np.argmax(pair_totals, axis=1)
best_pair_mm = pair_totals.max(axis=1)

windows = np.char.add(np.char.add(months[start_idx], "-"), months[start_idx + 1])

print("pair_totals shape:", pair_totals.shape)
for county, window, mm in zip(counties, windows, best_pair_mm):
    print(f"  {county:<9} {window:<16} ({mm} mm combined)")

pair_totals shape: (5, 11)
  Nairobi   April-May        (395 mm combined)
  Nakuru    April-May        (265 mm combined)
  Kisumu    April-May        (420 mm combined)
  Mombasa   April-May        (390 mm combined)
  Machakos  November-December (278 mm combined)


## 8. Write `rainfall_summary.csv` (with the `csv` module)

In [8]:
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for row in zip(counties, annual_totals, wettest_months, dry_counts, windows):
        county, total, wettest, dry, window = row
        writer.writerow([county, int(total), str(wettest), int(dry), str(window)])

# Read it back to confirm it is readable and has exactly five data rows
with open("rainfall_summary.csv", newline="") as f:
    rows = list(csv.reader(f))

print("Header:", rows[0])
print("Data rows:", len(rows) - 1)
for r in rows[1:]:
    print(r)
assert len(rows) - 1 == 5

Header: ['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
Data rows: 5
['Nairobi', '1045', 'April', '4', 'April-May']
['Nakuru', '916', 'April', '3', 'April-May']
['Kisumu', '1371', 'April', '0', 'April-May']
['Mombasa', '1075', 'May', '2', 'April-May']
['Machakos', '811', 'November', '6', 'November-December']


## Planting Advisory

For Nairobi, Nakuru, Kisumu and Mombasa, plant in **April–May**, when the long rains peak; April is the wettest month in Nairobi, Nakuru and Kisumu, and May is the wettest in Mombasa. Kisumu is the most reliable county, with 1,371 mm a year, no dry months and about 420 mm falling in its two-month planting window. For Machakos, the best window is **November–December** (the short rains), and farmers should expect six months below 50 mm, so water-harvesting and drought-tolerant crops are advisable there. In Nairobi, Nakuru and Mombasa, plan irrigation or mulching for the 2–4 dry months each year. Treat these dates as a guide only: confirm the onset of rains with the local forecast before sowing.

## Data Quality Notes

- **Two Machakos readings were invalid** (April `"N/A"` and September `""`). Both were replaced with 68 mm, the rounded mean of the 10 valid readings (67.5 mm). These 136 mm are about 17% of Machakos's annual total of 811 mm, so that total is an estimate.
- **The imputed April value affects the runner-up window.** April–May (68 + 140 = 208 mm) ranks second in Machakos. April would need to be above about 138 mm to overtake November–December (278 mm), so the recommendation is fairly robust, but the April–May ranking is uncertain.
- **The imputed September value affects the dry-month count.** 68 mm is not dry, so Machakos shows 6 dry months. If September was really below 50 mm, the true count would be 7.
- **Whitespace was not an error.** `" 20 "` converts cleanly to 20, so it was kept as a genuine reading.
- **Limits of the data:** one year of monthly totals for one station per county. This hides year-to-year variation and dry spells within a month, which matter for crops.